In [2]:
import pandas as pd
import json
import numpy as np
import requests
from tqdm.auto import tqdm
from config import URL, TOKEN
from datetime import datetime
import urllib3
from dateutil.relativedelta import relativedelta

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [6]:
headers = {
    "Authorization": f"Bearer {TOKEN}"
}

start_date = datetime(2017, 1, 1)
today = datetime.now()

all_data = {"entries": []}

# Calculate total number of months
total_months = (
    (today.year - start_date.year) * 12
    + today.month - start_date.month
    + 1
)

current = start_date

progress = tqdm(
    total=total_months,
    desc="Fetching months",
    unit="month"
)

while current < today:

    next_month = current + relativedelta(months=1)

    # Do not go beyond today
    month_end = min(next_month, today)

    date_from = current.strftime("%Y%m%d000000")
    date_to = month_end.strftime("%Y%m%d%H%M%S")

    itemCount = np.inf
    offset = 0
    steps = 2000

    while offset < itemCount:

        url = (
            f"{URL}assignment/query?"
            f"filter="
            f"assignmentType:in:24h;spring;first,"
            f"date:gte:{date_from},"
            f"date:lt:{date_to}"
            f"&sort=startDate.timeUtc"
            f"&limit={steps}"
            f"&skip={offset}"
        )

        response = requests.get(
            url,
            headers=headers,
            verify=False
        )

        if response.status_code != 200:
            tqdm.write(
                f"Error for {date_from} -> {date_to}: "
                f"{response.status_code}"
            )
            tqdm.write(response.text)
            break

        responseData = response.json()
        entries = responseData.get("entries", [])

        itemCount = responseData["pagingInfo"]["itemCount"]

        all_data["entries"].extend(entries)

        # Show cumulative number of collected assignments
        progress.set_postfix(
            collected=f"{len(all_data['entries']):,}",
            current_month=current.strftime("%Y-%m")
        )

        if len(entries) == 0:
            break

        offset += len(entries)

    # One progress step = one completed month
    progress.update(1)

    current = next_month

progress.close()

print(f"Total assignments collected: {len(all_data['entries']):,}")

Fetching months:   0%|          | 0/117 [00:00<?, ?month/s]

Error for 20250401000000 -> 20250501000000: 504

Error for 20250701000000 -> 20250801000000: 504

Error for 20250901000000 -> 20251001000000: 504

Error for 20251001000000 -> 20251101000000: 504

Error for 20251201000000 -> 20260101000000: 504

Error for 20260101000000 -> 20260201000000: 504

Error for 20260301000000 -> 20260401000000: 504

Error for 20260401000000 -> 20260501000000: 504

Error for 20260501000000 -> 20260601000000: 504

Error for 20260601000000 -> 20260701000000: 504

Error for 20260701000000 -> 20260801000000: 504

Error for 20260801000000 -> 20260901000000: 504

Error for 20260901000000 -> 20260904123743: 504

Total assignments collected: 167,604


In [7]:
exceptions = [
	"metadata",
	"locality",
	"postalcode",
	"contactoptions",
	"personFullName",
	"personFullNameNoTitle",
	"tenantid",
	"persontype",
	"adresse",
	"mailAddressing",
	"socialInsuranceNumber",
	"comments",
	"givenName",
	"familyName",
	"primaryEmailAddress",
	'addresses',
	'primaryPhoneNumber',
	'address',
	'residentialAddress',
	'address',
	'geoLocation',
	'billingAddress',
	'serviceAddress',
	'legalAddress',
	'businessAddress',
	'emailAddresses',
	'phoneNumbers',
	'personFullNameNoTitle',
	'personFullName',
	'mailAddressing',
	'postalAddress',
	'contractualAddressing',
	'caatsDateOfBirth',
	'gender',
	'locations',
	'bankDetails',
	'personStatus',
	"chapterId",
	"language",
	"ordinal",
	"sectionId",
	"academicTitlePrefix",
	"personNr",
	"content",
	"consecutiveNumber",
	'Klient:innen_Empfohlen_Ja',
	'Klient:innen_Erstrkontakt_erfassen_Ja',
	'Klient:innen_Bew_Ein_Ja',
	'abrech-akonto',
	'abrech-buerge-hinterlegt',
	'pers-visite-keine',
	'medizinische-delegation-hochgeladen',
	'medizinische-delegation-nicht-notwendig',
	'Klient:innen_Medizinische_Delegation_Ja',
	'pflegerische-delegation-hochgeladen',
	'pflegerische-delegation-nicht-notwendig',
	'Klient:innen_Pflegerische_Delegation_JA',
	"admin-beruf",
	'pflegevisite-letzte-date',
	'pflegevisite-letzte-dgkp',
	'pflegerisch-person',
	'erstkontakt-bemerkung',
	"admin-kooperationspartner",
	'birthName',
'confessionId',
'confession',
'nationalityId',
'placeOfBirth',
'paymentBlockReason',
'chamberOfCommerceMembershipNr',
'avatarFileId',
'empfohlen-von-category',
'visibilityConditionId',
'deutschkennt-bew-von',
'Kinder_Nein',
'covid-1',
'covid-2',
'covid-3',
'keine-kurse',
'agentur-zuletzt',
'orgRef',
'businesscase',
'comment',
'commentArrival',
'commentDeparture',
'region',
'contactInfo',
'busnessCaseStatus',
'transportOrgArrival',
'transportOrgDeparture',
'assigneeContactInfo',
'assigneeNationality',
'assigneOrgRef',
'assignmentType',
'businessCaseOrgRef',
'businessCaseType',
'assigneeOrgRef',
'hasOrder',
'businessCaseStatus',
"businessCaseStartDate",
"businessCaseEndDate",
"clientAddress",
"region",
"arrivalDate",
"departureDate",
"timeUtc",
"objectKey",
"categoryShortName",
"hasDisplayText",
"displayText",
"objectId",
"objectType",
"cycleLength"
]

exceptions = [x.lower() for x in exceptions]

In [8]:
def extractStatement(statement):
    field = {}
    if statement["statementId"].lower() in exceptions:
        return field

    match statement["answerScheme"]:
        case 1:
            field[statement["statementId"]] = statement["answerValue"]["displayText"]
        case 2:
            #do nothing
            field = {}
        case 3:
            field[statement["statementId"]] = statement["answerYesno"] == 2
        case 4:
            answers = statement["answers"]
            for answer in answers:
                if "isSelected" in answer.keys():
                    field[answer["answerId"]] = answer["isSelected"]
                else:
                    field[answer["answerId"]] = False

        case 5:
            answers = statement["answers"]
            for answer in answers:
                if "isSelected" in answer.keys():
                    field[answer["answerId"]] = answer["isSelected"]
                else:
                    field[answer["answerId"]] = False
        case 8:
            if "answerDateTime" in statement.keys():
                if "userLocalTime" in statement["answerDateTime"].keys():
                    field[statement["statementId"]] = statement["answerDateTime"]["userLocalTime"]
                elif "timeUtc" in statement["answerDateTime"].keys():
                    field[statement["statementId"]] = statement["answerDateTime"]["timeUtc"]
        case _:
            print(f'Found statement: {statement["answerScheme"]} - {statement["statementId"]} -  {statement}')


    return field
        

def returnEntry(entry):
    keys = entry.keys()
    finalFields = {}
    if "statementId" in keys:
        finalFields = finalFields | extractStatement(entry)
    else:
        for key in keys:
            
            if key.lower() in exceptions:
                continue
            
            typing = type(entry[key]).__name__
            match typing:
                case "str":
                    finalFields[key] = entry[key]
                    
                case "int":
                    finalFields[key] = entry[key]

                case "bool":
                    finalFields[key] = entry[key]

                case "float":
                    finalFields[key] = entry[key]
                    
                case "dict":
                    if key == "assignee":
                        finalFields[key] = entry[key]["objectId"]
                        continue
                    if "displayText" in entry[key].keys():
                        finalFields[key] = entry[key]["displayText"]
                        continue
                    if "userLocalTime" in entry[key].keys():
                        finalFields[key] = entry[key]["userLocalTime"]
                        continue
                    if "timeUtc" in entry[key].keys():
                        finalFields[key] = entry[key]["timeUtc"]
                        continue
                    else:
                        finalFields = finalFields | returnEntry(entry[key])
                    
                case "list":
                    if key == "clients":
                        finalFields[key] = entry[key][0]["objectId"]
                    else:
                        for item in entry[key]:
                            if type(item).__name__ == "dict":
                                finalFields = finalFields | returnEntry(item)
                        

                case _:
                    print(typing)


    
    return finalFields

In [9]:
totalData = []

if all_data["entries"] is not None:
    for entry in all_data["entries"]:
        test = returnEntry(entry)
        totalData.append(test)
        
df = pd.DataFrame(totalData)
df.drop_duplicates(inplace=True)

In [10]:
for col in df.columns:
    non_null = df[col].dropna()

    if len(non_null) > 0 and non_null.isin([True, False]).all():
        df[col] = df[col].fillna(False).astype(bool)

bool_cols = df.select_dtypes(include=["bool", "boolean"]).columns

df[bool_cols] = df[bool_cols].fillna(False)


df = df.replace(r'^\s*$', np.nan, regex=True)

df

C:\Users\npozdena\AppData\Local\Temp\ipykernel_23212\826111804.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)
C:\Users\npozdena\AppData\Local\Temp\ipykernel_23212\826111804.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)
C:\Users\npozdena\AppData\Local\Temp\ipykernel_23212\826111804.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead.

,id,startDate,endDate,assignee,clients,admin-packet,arrivalSelfOrganized,departureSelfOrganized,arrivalNoBilling,departureNoBilling,isFirstAssignment,billed
0,e88cdba8-fdb5-e9fe-635b-5edea54631e1,20160914000000,20170405000000,e44f66ba-7109-798d-a7b6-9b3c7f311967,e66b5fc6-1eb7-e558-8cfa-c21b59627cca,Sonstige,False,False,False,False,False,NaN
1,5b5db4fe-beca-f2a4-466c-46d83ea6837c,20161012000000,20170111000000,76715853-3f04-fb22-4148-56e0c9f99c06,a360b954-c224-9fed-8683-61ba4e96c82d,CD Perle,False,False,False,False,False,NaN
2,a04ab468-fe04-e7c6-f2fd-f9a23bda6a0b,20161025000000,20170110000000,3a128675-bd76-5d3f-08d4-ada5ce463d51,5ce3d5ab-c86e-e2be-f4e3-3e75985e0460,Sonstige,True,True,True,True,False,NaN
3,bc0ae33d-566c-4e19-0984-2dd733df5e35,20161102000000,20170104000000,2bb3d415-9e79-64f0-209a-026b5822455d,6fd526ac-4f30-143a-7f01-c529f01d6dc8,Sonstige,False,False,False,False,False,NaN
4,51519ea1-35ab-003a-29ac-6c404cc3e020,20161115000000,20170104000000,6cb49891-0fd8-3a44-eb29-2aa3e5921989,c90a9bad-6c54-d3c6-2974-3008ab5de4ce,Basis,False,False,False,False,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
167599,f142c5e7-2926-e6a7-843f-44c4e34b8481,20260227000000,20260320000000,fa3848f7-b19c-8fc7-d263-cfff72529e1b,9f4f5249-efba-d3ff-aa86-fb1ab61a8e03,Klassik,True,True,True,True,False,2.026032e+13
167600,f62a26c1-3eee-0db4-ffdf-5328592d41ac,20260227000000,20260403000000,b6df1b98-7462-abf5-2130-91387290775e,84c8f7c4-c751-1a6c-fb0d-fe622d8a758c,Premium,False,False,False,False,False,2.026040e+13
167601,fe8bcbba-33c9-c774-53be-0bb2da783cc9,20260227000000,20260327000000,a627f419-72f6-253c-d23e-694216b5559c,dae5040d-f691-9046-16ab-6cb73e727f87,Premium,False,False,False,False,True,2.026033e+13
167602,dd6c69c2-3479-50e3-db03-1e0a2cd9e2b5,20260228000000,20260321000000,ec7bfc1f-eeea-e207-0a1f-836675301cf7,6303c4b1-975e-2175-ad1b-24b37cca4d32,Premium,True,True,True,True,False,2.026032e+13


In [11]:
df.isna().sum()[df.isna().sum() > 0]

admin-packet    14045
billed          80926
dtype: int64

In [12]:
if "billed" in df.columns and df["billed"].nunique != 2:
	df["billed"] = df["billed"].notna()
if "admin-packet" in df.columns:
	df = pd.get_dummies(df, columns=["admin-packet"], prefix="paket", drop_first=True)


In [13]:
df

,id,startDate,endDate,assignee,clients,arrivalSelfOrganized,departureSelfOrganized,arrivalNoBilling,departureNoBilling,isFirstAssignment,...,paket_Diplomierte Fachkraft,paket_KH-Nachversorgung,paket_Klassik,paket_Kurzzeit-Urlaubsvertetung,paket_Palliativbegleitung,paket_Premium,paket_Sonstige,paket_ZZ_Pflege leicht,paket_ZZ_Pflegerin,paket_ZZ_Standard Plus
0,e88cdba8-fdb5-e9fe-635b-5edea54631e1,20160914000000,20170405000000,e44f66ba-7109-798d-a7b6-9b3c7f311967,e66b5fc6-1eb7-e558-8cfa-c21b59627cca,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
1,5b5db4fe-beca-f2a4-466c-46d83ea6837c,20161012000000,20170111000000,76715853-3f04-fb22-4148-56e0c9f99c06,a360b954-c224-9fed-8683-61ba4e96c82d,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,a04ab468-fe04-e7c6-f2fd-f9a23bda6a0b,20161025000000,20170110000000,3a128675-bd76-5d3f-08d4-ada5ce463d51,5ce3d5ab-c86e-e2be-f4e3-3e75985e0460,True,True,True,True,False,...,False,False,False,False,False,False,True,False,False,False
3,bc0ae33d-566c-4e19-0984-2dd733df5e35,20161102000000,20170104000000,2bb3d415-9e79-64f0-209a-026b5822455d,6fd526ac-4f30-143a-7f01-c529f01d6dc8,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
4,51519ea1-35ab-003a-29ac-6c404cc3e020,20161115000000,20170104000000,6cb49891-0fd8-3a44-eb29-2aa3e5921989,c90a9bad-6c54-d3c6-2974-3008ab5de4ce,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167599,f142c5e7-2926-e6a7-843f-44c4e34b8481,20260227000000,20260320000000,fa3848f7-b19c-8fc7-d263-cfff72529e1b,9f4f5249-efba-d3ff-aa86-fb1ab61a8e03,True,True,True,True,False,...,False,False,True,False,False,False,False,False,False,False
167600,f62a26c1-3eee-0db4-ffdf-5328592d41ac,20260227000000,20260403000000,b6df1b98-7462-abf5-2130-91387290775e,84c8f7c4-c751-1a6c-fb0d-fe622d8a758c,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
167601,fe8bcbba-33c9-c774-53be-0bb2da783cc9,20260227000000,20260327000000,a627f419-72f6-253c-d23e-694216b5559c,dae5040d-f691-9046-16ab-6cb73e727f87,False,False,False,False,True,...,False,False,False,False,False,True,False,False,False,False
167602,dd6c69c2-3479-50e3-db03-1e0a2cd9e2b5,20260228000000,20260321000000,ec7bfc1f-eeea-e207-0a1f-836675301cf7,6303c4b1-975e-2175-ad1b-24b37cca4d32,True,True,True,True,False,...,False,False,False,False,False,True,False,False,False,False


In [14]:
protected_cols = ["startDate", "endDate", "assignee", "clients"]

feature_cols = df.columns.drop(protected_cols)

keep_features = feature_cols[
    df[feature_cols].isna().mean() <= 0.10
]

# Keep protected columns + surviving features
df = df[protected_cols + keep_features.tolist()]

df

,startDate,endDate,assignee,clients,id,arrivalSelfOrganized,departureSelfOrganized,arrivalNoBilling,departureNoBilling,isFirstAssignment,...,paket_Diplomierte Fachkraft,paket_KH-Nachversorgung,paket_Klassik,paket_Kurzzeit-Urlaubsvertetung,paket_Palliativbegleitung,paket_Premium,paket_Sonstige,paket_ZZ_Pflege leicht,paket_ZZ_Pflegerin,paket_ZZ_Standard Plus
0,20160914000000,20170405000000,e44f66ba-7109-798d-a7b6-9b3c7f311967,e66b5fc6-1eb7-e558-8cfa-c21b59627cca,e88cdba8-fdb5-e9fe-635b-5edea54631e1,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
1,20161012000000,20170111000000,76715853-3f04-fb22-4148-56e0c9f99c06,a360b954-c224-9fed-8683-61ba4e96c82d,5b5db4fe-beca-f2a4-466c-46d83ea6837c,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,20161025000000,20170110000000,3a128675-bd76-5d3f-08d4-ada5ce463d51,5ce3d5ab-c86e-e2be-f4e3-3e75985e0460,a04ab468-fe04-e7c6-f2fd-f9a23bda6a0b,True,True,True,True,False,...,False,False,False,False,False,False,True,False,False,False
3,20161102000000,20170104000000,2bb3d415-9e79-64f0-209a-026b5822455d,6fd526ac-4f30-143a-7f01-c529f01d6dc8,bc0ae33d-566c-4e19-0984-2dd733df5e35,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
4,20161115000000,20170104000000,6cb49891-0fd8-3a44-eb29-2aa3e5921989,c90a9bad-6c54-d3c6-2974-3008ab5de4ce,51519ea1-35ab-003a-29ac-6c404cc3e020,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167599,20260227000000,20260320000000,fa3848f7-b19c-8fc7-d263-cfff72529e1b,9f4f5249-efba-d3ff-aa86-fb1ab61a8e03,f142c5e7-2926-e6a7-843f-44c4e34b8481,True,True,True,True,False,...,False,False,True,False,False,False,False,False,False,False
167600,20260227000000,20260403000000,b6df1b98-7462-abf5-2130-91387290775e,84c8f7c4-c751-1a6c-fb0d-fe622d8a758c,f62a26c1-3eee-0db4-ffdf-5328592d41ac,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
167601,20260227000000,20260327000000,a627f419-72f6-253c-d23e-694216b5559c,dae5040d-f691-9046-16ab-6cb73e727f87,fe8bcbba-33c9-c774-53be-0bb2da783cc9,False,False,False,False,True,...,False,False,False,False,False,True,False,False,False,False
167602,20260228000000,20260321000000,ec7bfc1f-eeea-e207-0a1f-836675301cf7,6303c4b1-975e-2175-ad1b-24b37cca4d32,dd6c69c2-3479-50e3-db03-1e0a2cd9e2b5,True,True,True,True,False,...,False,False,False,False,False,True,False,False,False,False


In [15]:
df.isna().sum()[df.isna().sum() > 0]

Series([], dtype: int64)

In [16]:
df.to_csv("assignments.csv", sep=";", index=False)